# MCP Types
1. Local MCP Servers
2. MCP Servers connecting to APIs
3. Remote MCP servers (rare case)

In [11]:
from dotenv import load_dotenv
from agents import Agent, Runner, trace
from agents.mcp import MCPServerStdio
import os
from IPython.display import Markdown, display
from datetime import datetime
load_dotenv(override=True)

True

### The first type of MCP Server: runs locally, everything local

Here's a really interesting one: a knowledge-graph based memory.

It's a persistent memory store of entities, observations about them, and relationships between them.

https://github.com/modelcontextprotocol/servers/tree/main/src/memory

Ensure the directory `memory` exists for storing information

In [7]:
params = {"command": "npx","args": ["-y", "mcp-memory-libsql"],"env": {"LIBSQL_URL": "file:./memory/storage.db"}}

async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()

mcp_tools

[Tool(name='create_entities', title='Create new entities with observations', description='Create new entities with observations', inputSchema={'type': 'object', 'properties': {'entities': {'type': 'array', 'items': {'type': 'object', 'properties': {'name': {'type': 'string'}, 'entityType': {'type': 'string'}, 'observations': {'type': 'array', 'items': {'type': 'string'}}}, 'required': ['name', 'entityType', 'observations']}}}, 'required': ['entities'], '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=None, icons=None, annotations=None, meta=None, execution=None),
 Tool(name='search_nodes', title='Search for entities and their relations using text search with relevance ranking', description='Search for entities and their relations using text search with relevance ranking', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string'}, 'limit': {'type': 'number'}}, 'required': ['query'], '$schema': 'http://json-schema.org/draft-07/schema#'}, outputSchema=No

In [8]:
instructions = "You use your entity tools as a persistent memory to store and recall information about your conversations."
request = "My name's Satish. I'm an LLM engineer. I'm learning about AI Agents, including the incredible MCP protocol. \
MCP is a protocol for connecting agents with tools, resources and prompt templates, and makes it easy to integrate AI agents with capabilities."
model = "gpt-4.1-mini"

In [ ]:
# Following will populate information from the request defined above
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

Nice to meet you, Satish! It's great that you're diving into AI Agents and the MCP protocol. If you have any questions about AI Agents, MCP, or anything related to your learning journey, feel free to ask!

In [10]:
# Check whether it can pull the information stored from previous call
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, "My name's Satish. What do you know about me?")
    display(Markdown(result.final_output))

I know that you are Satish, and you are an LLM engineer. Is there anything else you'd like me to remember about you?

Check the trace:

https://platform.openai.com/traces

### The 2nd type of MCP server - runs locally, calls a web service

### Tavily Search 

Set up tavily search account, and put your key in the .env under `TAVILY_API_KEY`

https://app.tavily.com



In [12]:
env = {"TAVILY_API_KEY": os.getenv("TAVILY_API_KEY")}
params = {"command": "npx", "args": ["-y", "tavily-mcp"], "env": env}
 
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as server:
    mcp_tools = await server.list_tools()
 
mcp_tools

[Tool(name='tavily_search', title=None, description='Search the web for current information on any topic. Use for news, facts, or data beyond your knowledge cutoff. Returns snippets and source URLs.', inputSchema={'type': 'object', 'properties': {'query': {'type': 'string', 'description': 'Search query'}, 'search_depth': {'type': 'string', 'enum': ['basic', 'advanced', 'fast', 'ultra-fast'], 'description': "The depth of the search. 'basic' for generic results, 'advanced' for more thorough search, 'fast' for optimized low latency with high relevance, 'ultra-fast' for prioritizing latency above all else", 'default': 'basic'}, 'topic': {'type': 'string', 'enum': ['general'], 'description': 'The category of the search. This will determine which of our agents will be used for the search', 'default': 'general'}, 'time_range': {'type': 'string', 'description': 'The time range back from the current date to include in the search results', 'enum': ['day', 'week', 'month', 'year']}, 'start_date':

In [13]:
instructions = "You are able to search the web for information and briefly summarize the takeaways."
request = f"Please research the latest news on Amazon stock price and briefly summarize its outlook. \
For context, the current date is {datetime.now().strftime('%Y-%m-%d')}"
model = "gpt-4.1-mini"

In [14]:
async with MCPServerStdio(params=params, client_session_timeout_seconds=30) as mcp_server:
    agent = Agent(name="agent", instructions=instructions, model=model, mcp_servers=[mcp_server])
    with trace("conversation"):
        result = await Runner.run(agent, request)
    display(Markdown(result.final_output))

As of March 20, 2026, Amazon's stock (AMZN) price is around $205 per share with a market capitalization of approximately $2.2 trillion. The stock has seen a slight recent decline but maintains a generally positive longer-term outlook. Analysts have price targets averaging around $280, with some targets as high as $360, and the latest analyst ratings generally reiterate a "Buy" stance. Amazon's valuation metrics such as trailing P/E (28.64) and forward P/E (25.84) indicate reasonable growth expectations.

Financially, Amazon reported strong revenue and earnings in FY25, with Q4 revenue exceeding $213 billion and earnings around $21 billion. The company continues to invest and grow in key areas like ultrafast delivery, AI shopping, and autonomous vehicle technologies (robotaxis). The AWS cloud segment is expected to grow significantly, potentially reaching $600 billion in revenue by 2036, supporting long-term growth.

Price forecasts suggest a potential rise to around $338 per share by 2030, reflecting expected revenue growth to $1.15 trillion. Overall, Amazon's stock outlook remains bullish with continued innovation and expansion, though some short-term price fluctuations are noted.

Let me know if you want a more detailed breakdown or specific recent news highlights.

### One more type 2 MCP server using polygon.io

Polygon.io is a hugely popular financial data provider. It has a free plan.  Signup, get a key, and add it to .env as `POLYGON_API_KEY`

https://polygon.io


In [15]:
load_dotenv(override=True)
polygon_api_key = os.getenv("POLYGON_API_KEY")
if not polygon_api_key:
    print("POLYGON_API_KEY is not set")

In [16]:
from polygon import RESTClient
client = RESTClient(polygon_api_key)
client.get_previous_close_agg("AAPL")[0]

PreviousCloseAgg(ticker='AAPL', close=247.99, high=249.1999, low=246, open=247.975, timestamp=1774036800000, volume=88752566.0, vwap=248.0116)